# 14장 FastAPI 수신 서버

이 노트북은 웹 교재 [14장 FastAPI 수신 서버](https://pidocs.aifusionlab.org/ai-python/ch14/)의 본문 코드를 절 순서대로 담고 있습니다. 위에서부터 차례로 실행하면서 본문의 실행 결과와 비교합니다.

## 실습 자료 내려받기

In [ ]:
!wget -q -nc https://raw.githubusercontent.com/sumilee-phd/pidocs/main/docs/data/class_rssi_20261104.csv
!ls *.csv

## 14.1 웹 API의 구조

[본문 보기](https://pidocs.aifusionlab.org/ai-python/ch14/14-1/)

In [ ]:
import json

m = {"node_id": "S01", "zone": "D1", "rssi": -55}
text = json.dumps(m)                 # 딕셔너리 → JSON 문자열
print(text, type(text))
back = json.loads(text)              # JSON 문자열 → 딕셔너리
print(back["rssi"] + 1, type(back))

## 14.2 FastAPI 엔드포인트

[본문 보기](https://pidocs.aifusionlab.org/ai-python/ch14/14-2/)

In [ ]:
import importlib.util, subprocess, sys

if importlib.util.find_spec("fastapi") is None:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "fastapi", "httpx"])
print("fastapi 준비:", importlib.util.find_spec("fastapi") is not None)

In [ ]:
from fastapi import FastAPI
from pydantic import BaseModel, Field
from statistics import median

class Measure(BaseModel):
    node_id: str = Field(min_length=1, max_length=16)   # 기기 번호
    zone: str = Field(pattern=r"^D[1-4]$")              # 측정 지점 D1~D4
    rssi: int = Field(ge=-100, le=0)                    # 신호 세기(dBm)

In [ ]:
app = FastAPI(title="piDocs 측정 서버")
records = []                                   # 수신 기록(서버를 다시 실행하면 초기화)

@app.post("/measure")
def post_measure(m: Measure):
    records.append(m.model_dump())             # 모델 → 딕셔너리
    return {"ok": True, "count": len(records)}

@app.get("/summary")
def get_summary():
    zones = sorted({r["zone"] for r in records})
    return {z: {"count": sum(1 for r in records if r["zone"] == z),
                "median": median(r["rssi"] for r in records if r["zone"] == z)}
            for z in zones}

In [ ]:
from fastapi.testclient import TestClient

api = TestClient(app)
r = api.post("/measure", json={"node_id": "S01", "zone": "D1", "rssi": -55})
print(r.status_code, r.json())
r = api.post("/measure", json={"node_id": "S01", "zone": "D9", "rssi": -55})
print(r.status_code, r.json()["detail"][0]["msg"])
r = api.post("/measure", json={"node_id": "S01", "zone": "D1", "rssi": 20})
print(r.status_code, r.json()["detail"][0]["msg"])
print(api.get("/summary").json())

## 14.3 판정 모듈 연결

[본문 보기](https://pidocs.aifusionlab.org/ai-python/ch14/14-3/)

In [ ]:
# judge.py: 7.1절 지점별 분위수 기준(11/4 수업 기록)
THRESH = {"D1": (-62.0, -56.0), "D2": (-77.0, -74.0),
          "D3": (-68.0, -66.0), "D4": (-73.3, -69.0)}   # (10% 분위수, 25% 분위수)

def judge_rssi(zone, v):
    """지점별 분위수 기준으로 신호 세기를 3단계 판정한다."""
    q10, q25 = THRESH[zone]
    if v < q10:
        return "경고"
    if v < q25:
        return "주의"
    return "정상"

print(judge_rssi("D1", -65), judge_rssi("D2", -65))

In [ ]:
@app.post("/judge")
def post_judge(m: Measure):
    level = judge_rssi(m.zone, m.rssi)
    records.append({**m.model_dump(), "level": level})
    return {"ok": True, "level": level}

r = api.post("/judge", json={"node_id": "S02", "zone": "D2", "rssi": -80})
print(r.status_code, r.json())

## 14.4 가상 노드로 서버 시험

[본문 보기](https://pidocs.aifusionlab.org/ai-python/ch14/14-4/)

In [ ]:
import time
import pandas as pd

records.clear()                                 # 앞 절의 시험 기록 삭제
rssi = pd.read_csv("class_rssi_20261104.csv")
codes = []
t0 = time.time()
for row in rssi.itertuples():
    r = api.post("/judge", json={"node_id": row.node, "zone": row.zone, "rssi": int(row.rssi)})
    codes.append(r.status_code)
print("전송", len(codes), "건, 상태 코드", pd.Series(codes).value_counts().to_dict())
print("저장", len(records), "건")

In [ ]:
print(api.get("/summary").json())
print(rssi.groupby("zone")["rssi"].median().to_dict())

saved = pd.DataFrame(records)
pd.crosstab(saved["zone"], saved["level"])[["정상", "주의", "경고"]]